# Task 1: Data Extraction, Cleaning, EDA, Stationarity, and Risk Metrics

This notebook covers the full Task 1 workflow for TSLA, BND, and SPY from 2015-01-01 through 2026-06-30.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import DATA_DIR, FIGURES_DIR, REPORTS_DIR
from src.data_loader import fetch_all_assets, pivot_adjusted_close, save_asset_data
from src.preprocessing import calculate_daily_returns, clean_all_assets, detect_return_outliers, inspect_data_quality
from src.risk import calculate_risk_table
from src.stationarity import run_adf_for_prices_and_returns
from src.visualization import plot_closing_prices, plot_daily_returns, plot_outliers, plot_rolling_statistics

## 1. Extract Data with YFinance
The end date is set to 2026-07-01 because Yahoo Finance treats the end date as exclusive, which includes 2026-06-30.

In [ ]:
raw_assets = fetch_all_assets()
cleaned_assets = clean_all_assets(raw_assets)
save_asset_data(cleaned_assets, DATA_DIR)
prices = pivot_adjusted_close(cleaned_assets)
prices.head()

## 2. Data Quality and Cleaning
Data types and missing values are inspected before and after cleaning. Missing price values are handled through time interpolation followed by forward/backward fill.

In [ ]:
quality = {ticker: inspect_data_quality(frame) for ticker, frame in cleaned_assets.items()}
quality["TSLA"]

## 3. EDA Visualizations

In [ ]:
returns = calculate_daily_returns(prices)
plot_closing_prices(prices, FIGURES_DIR / "task1_closing_prices.png");

In [ ]:
plot_daily_returns(returns, FIGURES_DIR / "task1_daily_returns.png");

In [ ]:
plot_rolling_statistics(prices, returns, ticker="TSLA", window=30, output_path=FIGURES_DIR / "task1_tsla_rolling_mean_volatility.png");

## 4. Outlier Detection
Return outliers are detected using absolute z-scores greater than or equal to 3.

In [ ]:
outliers = detect_return_outliers(returns, threshold=3.0)
outliers.to_csv(REPORTS_DIR / "return_outliers_zscore.csv", index=False)
plot_outliers(returns, outliers, ticker="TSLA", output_path=FIGURES_DIR / "task1_tsla_outliers.png");
outliers.head()

## 5. ADF Stationarity Tests
The ADF test is applied to adjusted closing prices and daily returns. Prices are often non-stationary, while daily returns are usually more stationary, which supports differencing in ARIMA models.

In [ ]:
adf_results = run_adf_for_prices_and_returns(prices, returns)
adf_results.to_csv(REPORTS_DIR / "adf_stationarity_results.csv", index=False)
adf_results

## 6. Risk Metrics
The table below reports historical 95% VaR, Sharpe Ratio, annualized return, and annualized volatility.

In [ ]:
risk_table = calculate_risk_table(returns)
risk_table.to_csv(REPORTS_DIR / "risk_metrics.csv")
risk_table